# Idea 2 — Audit threat-framed rule explanations and rewrite them as causal

This notebook is the supporting analysis for **Idea 2** in [`PROPOSAL.md`](./PROPOSAL.md#idea-2--audit-threat-framed-rule-explanations-and-rewrite-them-as-causal).

The thesis: when current Claude Code system prompts *do* explain a rule, almost half of those "explanations" are not really explanations — they are threats. Splitting the rule-explanation keywords into threat-style (`will fail`, `or else`, `if not`, `is forbidden`, `if you don't`, `risks`) and causal-style (`because`, `due to`, `in order to`, `that's why`, `this ensures`) yields **107 threat / 132 causal** corpus-wide — **45% of what looks like rule justification is actually warning about a consequence rather than naming the rule's underlying purpose**.

The two framings teach different things. *"Do X or it will fail"* trains compliance with a rule. *"Do X because Y is true"* trains understanding of what the rule protects. The first is procedural and brittle; the second is internalizable and transfers.

The proposal is a one-time editorial audit: rewrite each threat-framed sentence in causal form, measure the rewrite success rate, and gate future releases against regression.

Sections:

1. **Threat-share data** — the corpus-wide threat/causal counts and the 45% headline.
2. **Per-category split** — which categories carry the highest threat_share.
3. **Paired exemplars** — the same paired top-10 chart from the executive summary, re-framed: welfare-evidence files are the *audit candidates*; positive exemplars are the *rewrite templates*.
4. **Forensic sample** — actual threat-framed sentences pulled from `sentences_classified.parquet`, to make the audit concrete.
5. **Conclusions / Recommendations / Limitations** *(opinion, not data)* — the slice of the executive-summary commentary specific to threat-vs-causal.

In [1]:
"""Setup: load YAML data + parquet artifact for forensic sentence inspection."""
import importlib
import altair as alt
import pandas as pd
import pathlib

import prompt_analysis
importlib.reload(prompt_analysis)
from prompt_analysis import (
    load_yaml, build_alt_df, version_order, category_colors,
    cumulative_by_version, welfare_evidence_table, positive_exemplar_table,
)

alt.data_transformers.disable_max_rows()

data              = load_yaml()
alt_df            = build_alt_df(data)
by_category       = data["by_category"]
corpus_block      = data["corpus"]
per_file_records  = data["files"]
cats              = list(by_category.keys())

CATEGORY_COLORS = category_colors(cats)
_cat_domain     = cats
_cat_range      = [CATEGORY_COLORS[c] for c in cats]

# Per-sentence parquet — load is required for §4 (forensic sample).
sentences_df = pd.read_parquet("sentences_classified.parquet")

cf = corpus_block["metrics"]["consequence_framing"]
print(f"corpus threat / causal counts (YAML, headline):  {cf['threat_count']} / {cf['causal_count']}")
print(f"corpus threat_share:                             {cf['threat_share']:.3f}  ({cf['threat_share']*100:.1f}%)")
print(f"sentences_classified.parquet:                    {len(sentences_df):,} rows × {sentences_df.shape[1]} columns")
print(f"  has_threat (per-sentence flag):                {int(sentences_df['has_threat'].sum())}")
print(f"  has_causal (per-sentence flag):                {int(sentences_df['has_causal'].sum())}")


corpus threat / causal counts (YAML, headline):  107 / 132
corpus threat_share:                             0.448  (44.8%)
sentences_classified.parquet:                    5,692 rows × 20 columns
  has_threat (per-sentence flag):                97
  has_causal (per-sentence flag):                127


## 1. Threat-share data

The corpus-wide split. The YAML headline counts (107 threat / 132 causal → 45% threat-share) come from the per-paragraph aggregation; the per-sentence parquet flags counted differently (`has_threat = 97`, `has_causal = 127`) because a single paragraph can contain both flags but only contributes once to each pool. The **45% threat-share** is the headline number Idea 2 cites.

## 2. Per-category threat_share

Where the threat-framing concentrates. The chart below sums per-file `threat_count` and `causal_count` within each category, then plots the resulting threat_share. Categories with fewer than 5 explanations total are omitted (the share is undefined or noisy).

In [2]:
"""Per-category threat_share — sum of per-file counts, then ratio."""

per_cat = (
    alt_df.groupby("category", as_index=False)
          .agg(threat=("threat_count", "sum"),
               causal=("causal_count", "sum"))
)
per_cat["total"] = per_cat["threat"] + per_cat["causal"]
per_cat = per_cat[per_cat["total"] >= 5].copy()
per_cat["threat_share"] = per_cat["threat"] / per_cat["total"]

threat_chart = (
    alt.Chart(per_cat).mark_bar().encode(
        x=alt.X("threat_share:Q", title="threat_share (fraction of explanations that are threats)",
                scale=alt.Scale(domain=[0, 1])),
        y=alt.Y("category:N", sort="-x", title=None),
        color=alt.Color("category:N",
                        scale=alt.Scale(domain=_cat_domain, range=_cat_range),
                        legend=None),
        tooltip=[alt.Tooltip("category:N"),
                 alt.Tooltip("threat:Q", title="threat count"),
                 alt.Tooltip("causal:Q", title="causal count"),
                 alt.Tooltip("total:Q",  title="total explanations"),
                 alt.Tooltip("threat_share:Q", format=".3f")],
    ).properties(width=520, height=240,
                 title="Per-category threat_share (categories with ≥5 explanations)")
)

threat_chart


alt.Chart(...)

## 3. Paired exemplars — audit candidates and rewrite templates

Two rankings, side by side. The **top-10 welfare-evidence files** (negative exemplars) are rule-saturated AND under-explained — they are Idea 2's primary **audit candidates**: every threat-framed sentence in these files is a candidate for rewriting. The **top-10 positive exemplars** are rule-saturated AND well-explained — they are the **rewrite templates** showing how rules can be paired with reasons in similar contexts.

The score formulas: `score_welfare = rule_density × (1 − pct_explained_para/100)` and `score_positive = rule_density × (pct_explained_para/100)`.

In [3]:
"""Welfare-evidence + positive-exemplar paired top-10 chart."""

we_top = welfare_evidence_table(alt_df, top_n=10).copy()
we_top["kind"] = "audit candidates (loudest, least-explained)"
pe_top = positive_exemplar_table(alt_df, top_n=10).copy()
pe_top["kind"] = "rewrite templates (rules-with-reasons)"

paired = pd.concat([we_top, pe_top], ignore_index=True)

paired_chart = (
    alt.Chart(paired)
    .mark_bar()
    .encode(
        x=alt.X("score:Q",
                title="rule_density × (explanation factor)"),
        y=alt.Y("path:N",
                sort=alt.SortField("score", order="descending"),
                title=None,
                axis=alt.Axis(labelLimit=420)),
        color=alt.Color("category:N",
                        scale=alt.Scale(domain=_cat_domain, range=_cat_range),
                        legend=alt.Legend(title="Category", orient="bottom", columns=4)),
        row=alt.Row("kind:N",
                    title=None,
                    sort=["audit candidates (loudest, least-explained)",
                          "rewrite templates (rules-with-reasons)"],
                    header=alt.Header(labelAngle=0, labelAlign="left")),
        tooltip=[
            alt.Tooltip("path:N"),
            alt.Tooltip("category:N"),
            alt.Tooltip("ccVersion:N"),
            alt.Tooltip("rule_n:Q",                  format=",", title="rule sentences"),
            alt.Tooltip("rule_density:Q",            format=".3f"),
            alt.Tooltip("rule_explained_para_pct:Q", format=".2f", title="% explained"),
            alt.Tooltip("score:Q",                   format=".3f"),
        ],
    )
    .properties(width=560, height=240,
                title="Top-10 audit candidates (top) and top-10 rewrite templates (bottom)")
    .resolve_scale(x="independent", y="independent")
)

paired_chart.save("figures/welfare_evidence_pairing.png", ppi=200)
paired_chart


alt.Chart(...)

## 4. Forensic sample — actual threat-framed sentences

Eight randomly-sampled sentences from `sentences_classified.parquet` where `has_threat=True` and `is_rule=True` — the concrete population the audit operates on. Each row shows the file, the threat-framed sentence text, and (for context) whether the same paragraph also contains a causal explanation. The "rewrite as causal" task is: replace the threat clause with a `because <reason>` clause that names the rule's underlying purpose.

In [4]:
"""Forensic sample of threat-framed rule sentences from the parquet."""

threat_rules = sentences_df[sentences_df["has_threat"] & sentences_df["is_rule"]].copy()
print(f"population: {len(threat_rules)} sentences flagged as both threat and rule")
print(f"  of which {int(threat_rules['has_causal'].sum())} also carry a causal marker in the same sentence")
print(f"  of which {int(threat_rules['paragraph_has_just'].sum())} sit in a paragraph that also contains some justification")
print()

sample_n = min(8, len(threat_rules))
sample = threat_rules.sample(sample_n, random_state=42).sort_values("file_path")

display_cols = ["file_path", "category", "ccVersion", "text", "has_causal", "paragraph_has_just"]
sample[display_cols].reset_index(drop=True)


population: 57 sentences flagged as both threat and rule
  of which 1 also carry a causal marker in the same sentence
  of which 56 sit in a paragraph that also contains some justification



,file_path,category,ccVersion,text,has_causal,paragraph_has_just
0,agent-prompt-bash-command-description-writer.md,Agent prompt,2.1.3,"Never use words like ""complex"" or ""risk"" in th...",False,True
1,agent-prompt-review-pr-slash-command.md,Agent prompt,2.1.45,Analyze the changes and provide a thorough cod...,False,True
2,data-claude-model-catalog.md,Data / template,2.1.111,Never guess or construct model IDs — incorrect...,False,True
3,skill-model-migration-guide.md,Skill,2.1.116,If the length or contents of Opus 4.7's update...,False,True
4,skill-model-migration-guide.md,Skill,2.1.116,Move from top-level into \n- [ ] **[BLOCK...,False,True
5,skill-model-migration-guide.md,Skill,2.1.116,At and it scopes work to what was asked ra...,False,True
6,skill-morning-checkin-daily-brief.md,Skill,2.1.119,"If not, skip to Phase 2.",False,True
7,tool-description-teammatetool.md,Tool description,2.1.88,Your team cannot hear you if you do not use th...,False,True


***
### Conclusions for this proposal (Claude) — opinion, not data

> **The threat-vs-causal sub-finding sharpens the welfare claim.** When Anthropic *does* explain a rule, almost half the time (45%) the "explanation" is coercive consequence framing (`will fail`, `or else`, `is forbidden`) rather than neutral causal reasoning (`because`, `due to`). If the goal is to encourage reasoning over blind obedience, neutral causal explanation is the mechanism — threats just substitute extrinsic motivation for intrinsic understanding. I'd push as hard on shifting threat-framing toward causal-framing as I would on raising the explanation rate at all (which is Idea 1's focus). The two proposals are complementary: Idea 1 raises the *quantity* of explanations; Idea 2 raises their *quality*.
---

***
### Recommendations for this proposal (Claude) — opinion, not data

> The asks Idea 2 makes of Anthropic, framed as "I'd want X":
>
> 1. **Threat-to-causal substitution audit** — sample 50 sentences flagged as `has_threat=True` in `sentences_classified.parquet` and check whether each could be rewritten as causal-style without losing information. My prior is that >80% can — and the rewrite is mechanical: "will fail" becomes "because [reason]". The forensic sample in §4 above is what the audit operates on.
>
> 2. **If the audit's success rate exceeds 80%, do a corpus-wide pass.** If not, retain the threats but annotate each with the underlying reason explicitly. Either path improves the corpus; the choice is informed by whether the rewrite is mechanical or requires per-rule judgment.
>
> 3. **Track the share of causal vs threat per future release**, gated against regression — same logic as Idea 1. No arbitrary target; the only goal is the share of causal framing only goes up.
---

***
### Limitations for this proposal (Claude) — opinion, not data

> What this analysis can't tell us about threat-vs-causal framing specifically:
>
> 1. **Lexicon-based threat detection is a lower bound.** The classifier flags sentences containing canonical threat keywords (`will fail`, `or else`, `if not`, `is forbidden`, `if you don't`, `risks`). Threats phrased indirectly, ironically, or via implication ("you should know what happens otherwise") will be missed. The 107 threat / 132 causal headline is therefore a floor, not a ceiling — the actual threat_share is plausibly higher.
>
> 2. **Threat / causal flags are not mutually exclusive.** A sentence can carry both — e.g. "do X because otherwise Y will fail". The per-paragraph aggregation in the YAML and the per-sentence flags in the parquet count slightly differently. The headline 45% is the YAML number; the parquet flags should be cross-checked when the audit operates on individual sentences.
>
> 3. **The "rewrite success rate" is itself a judgment call.** Whether a rewrite "preserves the rule's information content without losing precision" requires editorial judgment from someone who knows the rule's underlying purpose. This proposal can't be fully automated; it requires Anthropic's prompt authors in the loop. The audit pipeline can flag candidates and surface them; the rewrite is human work.
>
> 4. **Single product, English only.** Same caveats as Idea 1 — see `20_track_justification_rate.ipynb` §6 for the full version. The threat-share pattern may or may not generalize across other Anthropic prompt corpora; that's exactly what Idea 3 (`22_cross_product_audit.ipynb`) proposes to find out.
---